# Análisis Biomecánico en Ciclismo en OpenPose
# Luis Nieto - Gabriela Osorio* 

## 0. Librerias e info general

In [ ]:
#importamos las librerias de interés
import os
import json
import math
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path

warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor':  '#ffffff',
    'axes.facecolor':    '#ffffff',
    'axes.edgecolor':    '#1a3a5c',
    'axes.labelcolor':   '#000000',
    'axes.titlecolor':   '#000000',
    'xtick.color':       '#000000',
    'ytick.color':       '#000000',
    'grid.color':        '#a4b0bc',
    'grid.alpha':        0.4,
    'text.color':        '#000000',
    'legend.facecolor':  '#ffffff',
    'legend.edgecolor':  '#1a3a5c',
    'figure.titlesize':  14,
    'axes.titlesize':    11,
    'axes.labelsize':    10,
    'font.family':       'monospace',
})

# Color para los gráficos y sujetos (los colores IEEE :P)
COLOR_INICIAL = '#00a3e0'  
COLOR_FATIGA  = '#78be20'  
COLORES_SUJETOS = ['#00a3e0', '#78be20', '#00629b', '#772583', '#009ca6']

# frames por segundo de los videos analizados
FPS = 29.97

# Ruta base del proyecto
BASE = Path('..') 
#Carpeta donde se guardarán los outputs de los análisis
OUTPUTS = BASE / 'outputs'

# Sujetos a analizar
SUJETOS = ['P1', 'P2', 'P3', 'P4', 'P5']
CONDICIONES = ['Inicial', 'Fatiga']

#Ruta donde quedarán los outputs
print(f'Ruta de outputs: {OUTPUTS.resolve()}')

## 1. Funciones de procesamiento de keypoints

In [ ]:
#Se definen las funciones para extraer y calcular los angulos articulares directamente desde los archivos JSON generados por openpose.

# 1. definimos la función para extraer los puntos (coordenadas x, y) y la confianza de cada keypoint
def extraer_punto(keypoints, indice):
    #calculamos la posición base multiplicado el indice por 3 (x, y, confianza)
    b = indice * 3
    return (keypoints[b], keypoints[b + 1], keypoints[b + 2])

# 2. función para calcular el angulo entre tres puntos (p1, p2, p3) con p2 como vértice
def calcular_angulo(p1, p2, p3, umbral_confianza=0.1):
    #Si algun punto tiene baja confianza, no se calcula el angulo
    if any(p[2] < umbral_confianza for p in [p1, p2, p3]):
        return None
    
    #Vectores desde p2 a p1 y p3
    v1 = (p1[0] - p2[0], p1[1] - p2[1])
    v2 = (p3[0] - p2[0], p3[1] - p2[1])
    #producto punto y magnitudes para calcular el angulo
    dot = v1[0] * v2[0] + v1[1] * v2[1]
    m1  = math.sqrt(v1[0]**2 + v1[1]**2)
    m2  = math.sqrt(v2[0]**2 + v2[1]**2)
    if m1 == 0 or m2 == 0:
        return None
    #Calculamos el ángulo usando arcoseno y lo convertimos a grados
    return round(math.degrees(math.acos(max(-1.0, min(1.0, dot / (m1 * m2))))), 2)

# 3. función para calcular el angulo del tronco respecto a la vertical usando hombro y cadera
def calcular_angulo_tronco(hombro, cadera, umbral_confianza=0.1):
    
    #Verificamos que ambos puntos tengan suficiente confianza
    if hombro[2] < umbral_confianza or cadera[2] < umbral_confianza:
        return None
    
    # Vector del tronco: de cadera a hombro
    vx = hombro[0] - cadera[0]
    vy = hombro[1] - cadera[1]
    # Vector vertical de referencia (apunta hacia arriba en coordenadas de imagen)
    # En OpenPose el eje Y crece hacia abajo, por eso usamos (0, -1)
    ref_x, ref_y = 0, -1
    mag = math.sqrt(vx**2 + vy**2)
    if mag == 0:
        return None
    dot = (vx * ref_x + vy * ref_y) / mag
    return round(math.degrees(math.acos(max(-1.0, min(1.0, dot)))), 2)

## 2. Carga de datos desde JSON

In [ ]:
# 4. función para seleccionar el ciclista correcto cuando hay multiples personas detectadas por OpenPose
def seleccionar_ciclista(personas, ancho_frame=1920):
    #Si solo hay una persona, la seleccionamos directamente
    if len(personas) == 1:
        return personas[0]['pose_keypoints_2d']
    
    #buscamos la persona más cercana al centro horizontal del frame
    centro_frame = ancho_frame / 2
    mejor_idx    = 0
    menor_dist   = float('inf')
    
    for idx, persona in enumerate(personas):
        kp = persona['pose_keypoints_2d']
        puntos_x = [
            kp[i * 3] for i in [1, 2, 5, 8, 9, 12]  # cuello, hombros, cadera
            if kp[i * 3 + 2] > 0.1 and kp[i * 3] > 0 # Solo puntos con buena confianza
        ]
        if not puntos_x:
            continue
        #promediamos las coordenadas x de los puntos seleccionados para estimar el centro horizontal de la persona
        centro_persona = sum(puntos_x) / len(puntos_x)
        dist = abs(centro_persona - centro_frame)
        #Guardamos la persona más cercana al centro del frame 
        if dist < menor_dist:
            menor_dist = dist
            mejor_idx  = idx
    
    return personas[mejor_idx]['pose_keypoints_2d']

# 5. función para cargar y procesar todos los JSON de un sujeto
def cargar_datos_sujeto(sujeto, condicion, outputs_dir, fps=29.97):
    
    #Construi la ruta a la carpeta JSON del sujeto 
    json_dir = outputs_dir / f'{sujeto}_{condicion}' / 'json'
    
    #Verificar que la capeta exista y tenga archivos JSON
    if not json_dir.exists():
        print(f'  No encontrado: {json_dir}')
        return None
    #Obtener la lista de archivos JSON ordenada por nombre
    archivos = sorted([
        f for f in os.listdir(json_dir)
        if f.endswith('_keypoints.json')
    ])
    
    if not archivos:
        print(f'  Sin archivos JSON en: {json_dir}')
        return None

    # Detectar ancho del frame desde el primer JSON valido
    ancho_frame = 1920  # default Full HD
    for nombre in archivos[:10]:
        try:
            with open(json_dir / nombre) as f:
                d = json.load(f)
            if d.get('people'):
                kp = d['people'][0]['pose_keypoints_2d']
                #buscamos el maximo valor de x para estimar el ancho 
                xs = [kp[i*3] for i in range(25) if kp[i*3] > 0]
                if xs:
                    ancho_frame = max(xs) * 1.1
                    break
        except Exception:
            continue
    #Procesamos cada frame del video
    filas = []
    for i, nombre_archivo in enumerate(archivos):
        ruta = json_dir / nombre_archivo
        try:
            with open(ruta, encoding='utf-8') as f:
                datos = json.load(f)
        except Exception:
            continue
        #s no se detecta ninguna persona, se agregan filas con valores nulos para ese frame
        if not datos.get('people'):
            filas.append({
                'Frame':     i,
                'Tiempo_s':  round(i / fps, 3),
                'Rodilla_I': None,
                'Cadera_I':  None,
                'Tronco':    None,
                'Tobillo_I': None,
                'Sujeto':    sujeto,
                'Condicion': condicion,
                'N_personas': 0,
            })
            continue
        
        # Seleccionar ciclista si hay multiples personas
        n_personas = len(datos['people'])
        kp = seleccionar_ciclista(datos['people'], ancho_frame)
        # Extraemos los puntos clave necesarios para calcular los ángulos
        p  = {n: extraer_punto(kp, n) for n in range(25)}
        
        #calculamos los ángulos de interés
        rodilla_i = calcular_angulo(p[12], p[13], p[14])
        cadera_i  = calcular_angulo(p[1],  p[12], p[13])
        tronco    = calcular_angulo_tronco(p[5], p[12])
        tobillo_i = calcular_angulo(p[13], p[14], p[19])
        
        #Guardamos los resultados del frame 
        filas.append({
            'Frame':      i,
            'Tiempo_s':   round(i / fps, 3),
            'Rodilla_I':  rodilla_i,
            'Cadera_I':   cadera_i,
            'Tronco':     tronco,
            'Tobillo_I':  tobillo_i,
            'Sujeto':     sujeto,
            'Condicion':  condicion,
            'N_personas': n_personas,
        })
    
    # Reportar frames con multiples personas detectadas
    df_temp = pd.DataFrame(filas)
    frames_multi = (df_temp['N_personas'] > 1).sum()
    if frames_multi > 0:
        print(f'  {sujeto}_{condicion}: {frames_multi} frames con multiples personas (corregidos)')
    
    return df_temp if filas else None


lista_dfs = []

for sujeto in SUJETOS:
    for condicion in CONDICIONES:
        df = cargar_datos_sujeto(sujeto, condicion, OUTPUTS, FPS)
        if df is not None:
            lista_dfs.append(df)
            print(f'  {sujeto}_{condicion}: {len(df)} frames cargados')

# DataFrame consolidado con todos los datos
df_total = pd.concat(lista_dfs, ignore_index=True)

print(f'\nTotal de frames cargados: {len(df_total)}')
print(f'Columnas: {list(df_total.columns)}')

## 3. Calidad de los datos

In [ ]:
#Lista de angulos a evaluar para la calidad de detección
angulos = ['Rodilla_I', 'Cadera_I', 'Tronco', 'Tobillo_I']

print('Porcentaje de frames validos por sujeto y condicion:')

#Calculo de la calidad de detección para cada sujeto y condición,
filas_calidad = []
for sujeto in SUJETOS:
    for condicion in CONDICIONES:
        #Filtramos el DataFrame para el sujeto y condición actual
        mask = (df_total['Sujeto'] == sujeto) & (df_total['Condicion'] == condicion)
        df_sub = df_total[mask]
        if df_sub.empty:
            continue
        #contamos el total de frames procesados
        total = len(df_sub)
        fila = {'Sujeto': sujeto, 'Condicion': condicion, 'Total_frames': total}
        #Para cada ángulo, calculamos el % de frames con valores no nulos
        for angulo in angulos:
            validos = df_sub[angulo].notna().sum()
            fila[f'{angulo}_pct'] = round(validos / total * 100, 1)
        filas_calidad.append(fila)

df_calidad = pd.DataFrame(filas_calidad)
print(df_calidad.to_string(index=False))

print('\nNota: valores menores a 80% indican problemas de deteccion en ese segmento.')

In [ ]:
# Creamos gráficos de barras para visualizar la calidad de detección
fig, axes = plt.subplots(2, 2, figsize=(12, 10))  # 2 filas x 2 columnas
fig.suptitle('Calidad de deteccion OpenPose por articulacion (%)', y=0.995)

# Preparamos las etiquetas y colores para cada barra
etiquetas = [f'{row.Sujeto}\n{row.Condicion}' for _, row in df_calidad.iterrows()]
colores_barra = [COLOR_INICIAL if row.Condicion == 'Inicial' else COLOR_FATIGA
                 for _, row in df_calidad.iterrows()]

# Aplanamos los ejes para iterar fácilmente (axes es una matriz 2x2)
axes_flat = axes.flatten()

# Creamos un gráfico para cada ángulo articular
for ax, angulo in zip(axes_flat, angulos):
    valores = df_calidad[f'{angulo}_pct'].values
    barras  = ax.bar(range(len(valores)), valores, color=colores_barra, width=0.6)
    
    # Línea horizontal de referencia en 80%
    ax.axhline(80, color='#cc2244', linestyle='--', linewidth=1, label='Umbral 80%')
    
    # Configuración del gráfico
    ax.set_title(angulo.replace('_', ' '))
    ax.set_ylabel('Frames validos (%)')
    ax.set_ylim(0, 105)
    ax.set_xticks(range(len(etiquetas)))
    ax.set_xticklabels(etiquetas, fontsize=7, rotation=45, ha='right')
    ax.legend(fontsize=8)
    ax.grid(axis='y')
    # Anotamos el porcentaje exacto sobre cada barra
    for j, v in enumerate(valores):
        ax.text(j, v + 1, f'{v}%', ha='center', va='bottom', fontsize=7)

# Agregamos leyenda de colores para diferenciar condiciones
from matplotlib.patches import Patch
leyenda = [
    Patch(facecolor=COLOR_INICIAL, label='Inicial'),
    Patch(facecolor=COLOR_FATIGA,  label='Fatiga'),
]
fig.legend(handles=leyenda, loc='upper right', bbox_to_anchor=(0.99, 0.99))

# Guardamos la figura
plt.tight_layout()
plt.savefig(OUTPUTS / 'figures' / 'calidad_deteccion.png',
            dpi=150, bbox_inches='tight', facecolor='#0a0f14')
plt.show()

# 3.1 Suavizado de la señal

In [ ]:
# =============================================================================
# SUAVIZADO DE SEÑAL - Filtro Butterworth de paso bajo
# Estandar en biomecánica (usado por Vicon, 3DMA y literatura de ciclismo)
# Frecuencia de corte: 6 Hz — adecuada para movimientos ciclicos lentos
# =============================================================================
from scipy.signal import butter, filtfilt

def butter_lowpass(serie, frecuencia_corte=6.0, fps=29.97, orden=4):
    # calculamos la frecuencia de Nyquist
    nyquist = fps / 2.0
    normal_cutoff = frecuencia_corte / nyquist
    #Diseñamos el filtro Butterworth
    b, a = butter(orden, normal_cutoff, btype='low', analog=False)
    
    # Interpolamos NaN para poder filtrar, luego los restauramos
    serie_interp = serie.interpolate(method='linear', limit_direction='both')
    nan_mask = serie.isna()
    
    # Necesitamos al menos 15 muestras para filtrar
    if serie_interp.notna().sum() < 15:
        return serie.values
    
    #Aplicamos filtfilt 
    filtrada = filtfilt(b, a, serie_interp.values)
    
    # Restaurar NaN originales
    filtrada[nan_mask] = np.nan
    return filtrada

#Aplicamos el filtro a todos los sujetos y angulos
for angulo in angulos:
    #Creamos una columna temporal para almacenar los valores filtrados
    df_total[f'{angulo}_filtrado'] = np.nan
    #Filtramos para cada sujeto y condición por separado
    for sujeto in SUJETOS:
        for condicion in CONDICIONES:
            mask = (df_total['Sujeto'] == sujeto) & (df_total['Condicion'] == condicion)
            if mask.sum() == 0:
                continue
            serie = df_total.loc[mask, angulo]
            df_total.loc[mask, f'{angulo}_filtrado'] = butter_lowpass(serie, fps=FPS)


#guardamos las originales con sufijo _raw por si necesitamos compararlas
for angulo in angulos:
    df_total[f'{angulo}_raw'] = df_total[angulo]
    df_total[angulo] = df_total[f'{angulo}_filtrado']
    df_total.drop(columns=[f'{angulo}_filtrado'], inplace=True)

print(f'Columnas disponibles: {list(df_total.columns)}')

## 4. Estadisticas descriptivas

Se calculan las estadisticas principales de cada angulo por sujeto y condicion:
media, desviacion estandar, minimo, maximo y rango de movimiento (ROM).

In [ ]:
#Calculamos estadisticas descriptivas para cada sujeto y condición
filas_stats = []

for sujeto in SUJETOS:
    for condicion in CONDICIONES:
        #Fultramos el DataFrame para el sujeto y condición actual
        mask   = (df_total['Sujeto'] == sujeto) & (df_total['Condicion'] == condicion)
        df_sub = df_total[mask]
        if df_sub.empty:
            continue
        fila = {'Sujeto': sujeto, 'Condicion': condicion}
        #Calculamos media, std, min, max y ROM para cada ángulo
        for angulo in angulos:
            serie = df_sub[angulo].dropna()
            if serie.empty:
                #Si no hay datos, none
                fila[f'{angulo}_media'] = None
                fila[f'{angulo}_std']   = None
                fila[f'{angulo}_min']   = None
                fila[f'{angulo}_max']   = None
                fila[f'{angulo}_ROM']   = None
            else:
                #Calculamos las estadisticas y las redondeamos a 2 decimales
                fila[f'{angulo}_media'] = round(serie.mean(), 2)
                fila[f'{angulo}_std']   = round(serie.std(), 2)
                fila[f'{angulo}_min']   = round(serie.min(), 2)
                fila[f'{angulo}_max']   = round(serie.max(), 2)
                fila[f'{angulo}_ROM']   = round(serie.max() - serie.min(), 2)
        filas_stats.append(fila)

df_stats = pd.DataFrame(filas_stats)

# Mostrar estadisticas por angulo
for angulo in angulos:
    cols = ['Sujeto', 'Condicion',
            f'{angulo}_media', f'{angulo}_std',
            f'{angulo}_min',   f'{angulo}_max', f'{angulo}_ROM']
    print(f'\n{angulo.replace("_", " ")} (grados):')
    print(df_stats[cols].to_string(index=False))

# Guardar en Excel
df_stats.to_excel(OUTPUTS / 'metrics' / 'estadisticas_descriptivas.xlsx', index=False)
print('\nEstadisticas guardadas en outputs/metrics/estadisticas_descriptivas.xlsx')

## 5. Visualizacion de angulos en el tiempo

Se grafica la serie temporal de cada angulo para cada sujeto,
superponiendo Inicial y Fatiga para comparacion directa.

In [ ]:
# Mapeo de nombres de ángulos al inglés
angulos_labels = {
    'Rodilla_I': 'Knee L',
    'Cadera_I':  'Hip L',
    'Tronco':    'Trunk',
    'Tobillo_I': 'Ankle L',
}

condiciones_labels = {
    'Inicial': 'Normal',
    'Fatiga':  'Fatigue',
}

for sujeto in SUJETOS:
    fig, axes = plt.subplots(4, 1, figsize=(14, 10))
    fig.suptitle(f'Subject: {sujeto} - Joint angle time series')
    
    for ax, angulo in zip(axes, angulos):
        for condicion, color in zip(CONDICIONES, [COLOR_INICIAL, COLOR_FATIGA]):
            mask = (df_total['Sujeto'] == sujeto) & (df_total['Condicion'] == condicion)
            df_sub = df_total[mask][['Tiempo_s', angulo]].dropna()
            if df_sub.empty:
                continue
            ax.plot(
                df_sub['Tiempo_s'],
                df_sub[angulo],
                color=color,
                linewidth=1.5,          # era 0.8
                alpha=0.85,
                label=condiciones_labels.get(condicion, condicion)
            )
        ax.set_ylabel('Angle (degrees)')
        ax.set_title(angulos_labels.get(angulo, angulo.replace('_', ' ')))
        ax.legend(fontsize=9)
        ax.grid(True)
    
    axes[-1].set_xlabel('Time (seconds)')
    plt.tight_layout()
    ruta_fig = OUTPUTS / 'figures' / f'{sujeto}_series_temporales.png'
    plt.savefig(ruta_fig, dpi=150, bbox_inches='tight', facecolor='#ffffff')
    plt.show()
    print(f'Figura guardada: {ruta_fig.name}')

## 6. Comparacion Inicial vs Fatiga

Se comparan las medias de cada angulo entre condicion Inicial y Fatiga
usando boxplots para visualizar la distribucion completa.

In [ ]:
#Creamos graficos boxplot para comparar inicial vs fatiga
fig, axes = plt.subplots(1, 4, figsize=(16, 6))
fig.suptitle('Distribucion de angulos: Inicial vs Fatiga (todos los sujetos)')

for ax, angulo in zip(axes, angulos):
    #Extraemos los datos de cada condición para el ángulo actual, eliminando NaN
    datos_inicial = df_total[df_total['Condicion'] == 'Inicial'][angulo].dropna()
    datos_fatiga  = df_total[df_total['Condicion'] == 'Fatiga'][angulo].dropna()
    #Creamos el boxplot con colores personalizados
    bp = ax.boxplot(
        [datos_inicial, datos_fatiga],
        labels=['Inicial', 'Fatiga'],
        patch_artist=True,
        medianprops=dict(color='#F0F2F5', linewidth=2),
        whiskerprops=dict(color='#8892A4'),
        capprops=dict(color='#8892A4'),
        flierprops=dict(marker='o', color='#4A5568', markersize=2, alpha=0.4),
    )
    #coloreamos las cajas según la condición
    bp['boxes'][0].set_facecolor(COLOR_INICIAL + '66')  # alpha hex
    bp['boxes'][1].set_facecolor(COLOR_FATIGA  + '66')
    bp['boxes'][0].set_edgecolor(COLOR_INICIAL)
    bp['boxes'][1].set_edgecolor(COLOR_FATIGA)
    
    # Anotar medias
    for j, datos in enumerate([datos_inicial, datos_fatiga], 1):
        media = datos.mean()
        ax.text(j, media, f' {media:.1f}', va='center', fontsize=8,
                color='#F0F2F5')
    
    ax.set_title(angulo.replace('_', ' '))
    ax.set_ylabel('Angulo (grados)')
    ax.grid(axis='y')

plt.tight_layout()
plt.savefig(OUTPUTS / 'figures' / 'comparacion_inicial_fatiga.png',
            dpi=150, bbox_inches='tight', facecolor='#0a0f14')
plt.show()
print('Figura guardada en outputs/figures/comparacion_inicial_fatiga.png')

## 7. Comparacion por sujeto: media de angulos Inicial vs Fatiga

Se visualiza la diferencia entre condiciones para cada sujeto individualmente,
lo que permite identificar patrones de respuesta a la fatiga.

In [ ]:
# Graficos de barras para comparar la media de cada ángulo por sujeto y condición
fig, axes = plt.subplots(1, 4, figsize=(16, 6))
fig.suptitle('Media de angulos por sujeto: Inicial vs Fatiga')

# Configuramos las posiciones de las barras para cada sujeto
x = np.arange(len(SUJETOS))
ancho = 0.35

for ax, angulo in zip(axes, angulos):
    medias_inicial = []
    medias_fatiga  = []
    #alculamos la media de cada ángulo para cada sujeto y condición
    for sujeto in SUJETOS:
        for condicion, lista in zip(CONDICIONES, [medias_inicial, medias_fatiga]):
            mask  = (df_total['Sujeto'] == sujeto) & (df_total['Condicion'] == condicion)
            serie = df_total[mask][angulo].dropna()
            lista.append(serie.mean() if not serie.empty else 0)
    # Creamos las barras para cada condición 
    barras1 = ax.bar(x - ancho/2, medias_inicial, ancho,
                     label='Inicial', color=COLOR_INICIAL, alpha=0.85)
    barras2 = ax.bar(x + ancho/2, medias_fatiga,  ancho,
                     label='Fatiga',  color=COLOR_FATIGA,  alpha=0.85)
    
    ax.set_title(angulo.replace('_', ' '))
    ax.set_ylabel('Media (grados)')
    ax.set_xticks(x)
    ax.set_xticklabels(SUJETOS, rotation=30, ha='right', fontsize=9)
    ax.legend(fontsize=9)
    ax.grid(axis='y')

plt.tight_layout()
plt.savefig(OUTPUTS / 'figures' / 'medias_por_sujeto.png',
            dpi=150, bbox_inches='tight', facecolor='#0a0f14')
plt.show()
print('Figura guardada en outputs/figures/medias_por_sujeto.png')

## 8. Rango de movimiento (ROM) por sujeto y condicion

In [ ]:
#cambios en el ROM entre Inicial y Fatiga pueden indicar compensaciones posturales o reduccion de eficiencia mecanica.
print('Rango de movimiento (ROM) en grados:')

cols_rom = ['Sujeto', 'Condicion'] + [f'{a}_ROM' for a in angulos]
print(df_stats[cols_rom].to_string(index=False))

# Diferencia de ROM entre Fatiga e Inicial por sujeto
print('\nDiferencia de ROM (Fatiga - Inicial):')

for sujeto in SUJETOS:
    # Obtenemos los datos de ambas condiciones para el sujeto actual
    fila_ini = df_stats[(df_stats['Sujeto'] == sujeto) & (df_stats['Condicion'] == 'Inicial')]
    fila_fat = df_stats[(df_stats['Sujeto'] == sujeto) & (df_stats['Condicion'] == 'Fatiga')]
    if fila_ini.empty or fila_fat.empty:
        continue
    print(f'\n  {sujeto}:')
    #Calculamos la diferencia de ROM para cada ángulo
    for angulo in angulos:
        rom_ini = fila_ini[f'{angulo}_ROM'].values[0]
        rom_fat = fila_fat[f'{angulo}_ROM'].values[0]
        if rom_ini and rom_fat:
            diff = round(rom_fat - rom_ini, 2)
            signo = '+' if diff >= 0 else ''
            print(f'    {angulo.replace("_", " "):12s}: {signo}{diff} grados')

# 9. Coeficiente de variación

In [ ]:
# COEFICIENTE DE VARIACION (CV)
# CV = (desviacion estandar / media) * 100
# Un CV mayor en fatiga indica mayor irregularidad en el pedaleo

print('Coeficiente de variacion (CV) por sujeto y condicion:')
#Calculamos el CV para cada ángulo, sujeto y condición
filas_cv = []
for sujeto in SUJETOS:
    for condicion in CONDICIONES:
        mask   = (df_total['Sujeto'] == sujeto) & (df_total['Condicion'] == condicion)
        df_sub = df_total[mask]
        fila   = {'Sujeto': sujeto, 'Condicion': condicion}
        for angulo in angulos:
            serie = df_sub[angulo].dropna()
            #El CV solo se calcula si hay datos y la media no es cero
            if not serie.empty and serie.mean() != 0:
                fila[f'{angulo}_CV'] = round(serie.std() / abs(serie.mean()) * 100, 2)
            else:
                fila[f'{angulo}_CV'] = None
        filas_cv.append(fila)

df_cv = pd.DataFrame(filas_cv)
print(df_cv.to_string(index=False))

# Diferencia de CV entre condiciones
print('\nDiferencia de CV (Fatiga - Inicial):')
for sujeto in SUJETOS:
    #Obtenemos los CV de ambas condiciones para el sujeto actual
    fila_ini = df_cv[(df_cv['Sujeto'] == sujeto) & (df_cv['Condicion'] == 'Inicial')]
    fila_fat = df_cv[(df_cv['Sujeto'] == sujeto) & (df_cv['Condicion'] == 'Fatiga')]
    if fila_ini.empty or fila_fat.empty:
        continue
    print(f'\n  {sujeto}:')
    for angulo in angulos:
        cv_ini = fila_ini[f'{angulo}_CV'].values[0]
        cv_fat = fila_fat[f'{angulo}_CV'].values[0]
        if cv_ini and cv_fat:
            diff   = round(cv_fat - cv_ini, 2)
            signo  = '+' if diff >= 0 else ''
            #Marcamos con un asterisco si la diferencia es mayor a 3%, como señal de posible irregularidad
            estado = '  <- mayor irregularidad en fatiga' if diff > 3 else ''
            print(f'    {angulo.replace("_", " "):12s}: {signo}{diff}%{estado}')

# Guardar
df_cv.to_excel(OUTPUTS / 'metrics' / 'coeficiente_variacion.xlsx', index=False)
print('\nGuardado en outputs/metrics/coeficiente_variacion.xlsx')